In [ ]:
%pip install torchmetrics
%pip install albumentations

: 

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from torch.utils.data import DataLoader, Dataset
import torch
import torch.nn as nn
import torch.optim as optim
import torchmetrics
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os
import warnings
import glob
from tqdm import tqdm
warnings.filterwarnings('ignore')
import joblib

In [ ]:
import os

# Dùng absolute path dựa trên vị trí notebook
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
dataset_dir = os.path.join(NOTEBOOK_DIR, "FruitVision")

# Fallback: thử các đường dẫn phổ biến
if not os.path.isdir(dataset_dir):
    for candidate in [
        "/home/long/Documents/Nhap_mon_thi_giac/Do_An/FruitVision",
        "FruitVision",
        "Dataset/archive/FruitVision",
        "Dataset\\archive\\FruitVision",
    ]:
        if os.path.isdir(candidate):
            dataset_dir = candidate
            break

print(f"Dataset dir: {dataset_dir}")
assert os.path.isdir(dataset_dir), f"Không tìm thấy thư mục dataset: {dataset_dir}"

fruit2id = {
    'apple': 0, 'banana': 1, 'grape': 2, 'orange': 3
}

valid_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

img_path = []
labels = []

for fruit_quality_name in os.listdir(dataset_dir):
    fruit_quality_folder_path = os.path.join(dataset_dir, fruit_quality_name)

    if os.path.isdir(fruit_quality_folder_path):
        folder_lower = fruit_quality_name.lower()

        fresh_id = 1
        if 'fresh' in folder_lower:
            fresh_id = 0

        fruit_id = -1
        for fruit_name, label in fruit2id.items():
            if fruit_name in folder_lower:
                fruit_id = label
                break

        if fruit_id == -1:
            continue

        for img_file in os.listdir(fruit_quality_folder_path):
            full_path = os.path.join(fruit_quality_folder_path, img_file)
            ext = os.path.splitext(img_file)[1].lower()
            if os.path.isfile(full_path) and ext in valid_exts:
                img_path.append(full_path)
                labels.append((fruit_id, fresh_id))

print(f"Total images found: {len(img_path)}")

# ====== CHIA DỮ LIỆU: train 70% / val 15% / test 15% ======
from sklearn.model_selection import train_test_split

# Tạo label ghép để stratify theo cả fruit_id và fresh_id
combined_labels = [f"{l[0]}_{l[1]}" for l in labels]

# Split: 70% train, 30% temp
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    img_path, labels, test_size=0.3, random_state=42, stratify=combined_labels
)

# Split temp: 50/50 → val 15%, test 15%
combined_temp = [f"{l[0]}_{l[1]}" for l in temp_labels]
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, random_state=42, stratify=combined_temp
)

# ====== GIỚI HẠN KÍCH THƯỚC TRAIN SET (sau split, trước training) ======
MAX_TRAIN_IMAGES = 40000
if len(train_paths) > MAX_TRAIN_IMAGES:
    combined_train = [f"{l[0]}_{l[1]}" for l in train_labels]
    train_paths, _, train_labels, _ = train_test_split(
        train_paths,
        train_labels,
        train_size=MAX_TRAIN_IMAGES,
        random_state=42,
        stratify=combined_train
    )
    print(f"Capped train set to {len(train_paths)} images (MAX_TRAIN_IMAGES={MAX_TRAIN_IMAGES})")
else:
    print(f"Train set giữ nguyên {len(train_paths)} images (<= {MAX_TRAIN_IMAGES})")

print(f"Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")
print(f"No overlap check: {len(set(train_paths) & set(val_paths))} train∩val, "
      f"{len(set(train_paths) & set(test_paths))} train∩test, "
      f"{len(set(val_paths) & set(test_paths))} val∩test")

# Kiểm tra 3 đường dẫn đầu tiên có tồn tại không
for p in train_paths[:3]:
    print(f"  ✓ {p} exists={os.path.isfile(p)}")


In [ ]:
class FruitData(Dataset):
  def __init__(self, img_path, labels, transform=None):
    self.img_path = img_path
    self.labels = labels
    self.transform = transform

  def __len__(self):
    return len(self.img_path)

  def __getitem__(self, idx):
    img_path = self.img_path[idx]
    img = cv2.imread(img_path)

    if img is None:
        raise FileNotFoundError(f"Không thể đọc ảnh: {img_path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    label = self.labels[idx]
    y1, y2 = label

    if self.transform is not None:
      augmented = self.transform(image=img)
      image = augmented['image']

    return image, torch.tensor(y1, dtype=torch.long), torch.tensor(y2, dtype=torch.long)


In [ ]:
# def get_transform():
#   train_trans = A.Compose([
#       A.Resize(224, 224),

#       # Augmentation nhẹ, phù hợp bài toán trái cây
#       A.HorizontalFlip(p=0.5),
#       A.ShiftScaleRotate(shift_limit=0.03, scale_limit=0.05, rotate_limit=10, p=0.3),

#       A.Normalize(
#           mean=[0.485, 0.456, 0.406],
#           std=[0.229, 0.224, 0.225],
#           max_pixel_value=255.0,
#           p=1.0
#       ),
#       ToTensorV2()
#   ])
#   val_trans = A.Compose([
#       A.Resize(224, 224),
#       A.Normalize(
#           mean=[0.485, 0.456, 0.406],
#           std=[0.229, 0.224, 0.225],
#           max_pixel_value=255.0,
#           p=1.0
#       ),
#       ToTensorV2()
#   ])
#   return train_trans, val_trans
def get_transform():
  train_trans = A.Compose([
      A.Resize(224, 224),

      A.HorizontalFlip(p=0.5),
      A.VerticalFlip(p=0.5),
      A.RandomRotate90(p=0.5),
      A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
      A.GaussNoise(var_limit=(10.0, 20.0), p=0.2),
      A.Normalize(
          mean=[0.485, 0.456, 0.406],
          std=[0.229, 0.224, 0.225],
          max_pixel_value=255.0,
          p=1.0
      ),
      ToTensorV2()
  ])
  val_trans = A.Compose([
      A.Resize(224, 224),
      A.Normalize(
          mean=[0.485, 0.456, 0.406],
          std=[0.229, 0.224, 0.225],
          max_pixel_value=255.0,
          p=1.0
      ),
      ToTensorV2()
  ])
  return train_trans, val_trans

In [ ]:
class BasicBlock(nn.Module):
  expansion = 1 

  def __init__(self, in_channels, out_channels, stride=1):
    super(BasicBlock, self).__init__()
    self.conv1 = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )
    self.conv2 = nn.Sequential(
        nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(out_channels)
    )
    self.relu = nn.ReLU(inplace=True)
    self.downsample = nn.Sequential()
    if stride != 1 or in_channels != self.expansion * out_channels:
      self.downsample = nn.Sequential(
          nn.Conv2d(in_channels, self.expansion * out_channels, kernel_size=1, stride=stride, bias=False),
          nn.BatchNorm2d(self.expansion * out_channels)
      )

  def forward(self, X):
    residual = X
    out = self.conv1(X)
    out = self.conv2(out)
    if self.downsample is not None:
      residual = self.downsample(X)
    out += residual
    out = self.relu(out)
    return out

In [ ]:
class ResNetNN(nn.Module):
  def __init__(self, block, num_blocks, num_classes=8):
    super(ResNetNN, self).__init__()
    self.in_channels = 64

    self.conv2 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
    self.bn = nn.BatchNorm2d(64)
    self.relu = nn.ReLU(inplace=True)
    self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

    self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
    self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
    self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
    self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)

    self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

  def _make_layer(self, block, out_channels, num_blocks, stride):
    stride = [stride] + [1] * (num_blocks - 1)
    layers = []
    for s in stride:
      layers.append(block(self.in_channels, out_channels, s))
      self.in_channels = block.expansion * out_channels

    return nn.Sequential(*layers)

  def forward(self, X):
    out = self.conv2(X)
    out = self.bn(out)
    out = self.relu(out)
    out = self.maxpool(out)

    out = self.layer1(out)
    out = self.layer2(out)
    out = self.layer3(out)
    out = self.layer4(out)

    out = self.avgpool(out)
    out = torch.flatten(out, 1)
    return out

In [ ]:
train_trans, val_trans = get_transform()

# ===== MỖI TẬP DÙNG DATA RIÊNG - KHÔNG OVERLAP =====
train = FruitData(train_paths, train_labels, transform=train_trans)
val   = FruitData(val_paths,   val_labels,   transform=val_trans)
test  = FruitData(test_paths,  test_labels,  transform=val_trans)

train_loader = DataLoader(train, batch_size=32, shuffle=True)
val_loader   = DataLoader(val,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test,  batch_size=32, shuffle=False)

print(f"Train loader: {len(train_loader)} batches ({len(train)} images)")
print(f"Val loader:   {len(val_loader)} batches ({len(val)} images)")
print(f"Test loader:  {len(test_loader)} batches ({len(test)} images)")


In [ ]:
class ModelResNet(nn.Module):
  def __init__(self):
    super(ModelResNet, self).__init__()
    self.alpha = 0.7
    self.device = "cuda" if torch.cuda.is_available() else "cpu"

    self.base = ResNetNN(BasicBlock, [2, 2, 2, 2])

    # self.block1 = nn.Sequential(  # cần check lại
    #   nn.Linear(512, 256), 
    #   nn.ReLU(),
    #   nn.Dropout(0.2),
    #   nn.Linear(256, 128)
    # )

    self.block1 = nn.Sequential(
      nn.Linear(512, 64),
      nn.ReLU(),
      nn.Dropout(0.2),
      nn.Linear(64, 4)
    )

    self.block2 = nn.Sequential(
      nn.Linear(512, 64),
      nn.ReLU(),
      nn.Dropout(0.2),
      nn.Linear(64, 2)
    )

    self.optimizer = torch.optim.Adam([
      {'params': self.base.parameters(), 'lr': 1e-5},
      {'params': self.block1.parameters(), 'lr': 3e-4},
      {'params': self.block2.parameters(), 'lr': 3e-4}
    ])

    self.criterion = nn.CrossEntropyLoss()
    self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    self.fruit_acc = torchmetrics.Accuracy(task='multiclass', num_classes=4).to(self.device)
    self.qual_acc = torchmetrics.Accuracy(task='multiclass', num_classes=2).to(self.device)


  def forward(self, X):
    X = self.base(X)
    y1, y2 = self.block1(X), self.block2(X)
    return y1, y2


  def train_step(self, X, y1, y2):
    self.train()
    self.optimizer.zero_grad()
    y1_pred, y2_pred = self.forward(X)
    loss1 = self.criterion(y1_pred, y1)
    loss2 = self.criterion(y2_pred, y2)
    loss = loss1 + loss2 # cần check lại
    loss.backward()
    self.optimizer.step()
    fruit_accu = self.fruit_acc(y1_pred.argmax(dim=1), y1)
    qual_accu = self.qual_acc(y2_pred.argmax(dim=1), y2)
    return loss.item(), fruit_accu.item(), qual_accu.item()

  def val_step(self, X, y1, y2):
    self.eval()
    with torch.no_grad():
      y1_pred, y2_pred = self.forward(X)
      loss1 = self.criterion(y1_pred, y1)
      loss2 = self.criterion(y2_pred, y2)
      loss = loss1 + loss2 # cần check lại
      fruit_accu = self.fruit_acc(y1_pred.argmax(dim=1), y1)
      qual_accu = self.qual_acc(y2_pred.argmax(dim=1), y2)
    return loss.item(), fruit_accu.item(), qual_accu.item()

  def train_model(self, epochs=10):
    for ep in range(epochs):
      self.train()
      train_loss, train_fruit, train_qual = 0, 0, 0
      val_loss, val_fruit, val_qual = 0, 0, 0
      for X, y1, y2 in tqdm(train_loader):
        X, y1, y2 = X.to(self.device), y1.to(self.device), y2.to(self.device)
        loss, fruit_acc, qual_acc = self.train_step(X, y1, y2)
        train_loss += loss
        train_fruit += fruit_acc
        train_qual += qual_acc

      self.eval()
      for X, y1, y2 in tqdm(val_loader):
        X, y1, y2 = X.to(self.device), y1.to(self.device), y2.to(self.device)
        loss, fruit_acc, qual_acc = self.val_step(X, y1, y2)
        val_loss += loss
        val_fruit += fruit_acc
        val_qual += qual_acc

      train_loss /= len(train_loader)
      train_fruit /= len(train_loader)
      train_qual /= len(train_loader)

      val_loss /= len(val_loader)
      val_fruit /= len(val_loader)
      val_qual /= len(val_loader)

      print(f"Epoch {ep+1}/{epochs}")
      print(f"Train Loss: {train_loss:.4f}, Train Fruit Acc: {train_fruit:.4f}, Train Qual Acc: {train_qual:.4f}")
      print(f"Val Loss: {val_loss:.4f}, Val Fruit Acc: {val_fruit:.4f}, Val Qual Acc: {val_qual:.4f}")


In [ ]:
model = ModelResNet()
model.to(model.device)
model.train_model(epochs=5)

## Other methods

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import random

# Từ điển ngược để ánh xạ ID về tên nhãn
id2fruit = {0: 'Apple', 1: 'Banana', 2: 'Grape', 3: 'Orange'}
id2fresh = {0: 'Fresh', 1: 'Rotten'}

def visualize_random_and_wrong_predictions(model, dataset, num_correct=3, num_wrong=3):
    model.eval()
    
    # Tạo danh sách các chỉ mục và xáo trộn ngẫu nhiên để tránh bị trùng lặp 1 quả
    indices = list(range(len(dataset)))
    random.shuffle(indices)
    
    correct_samples = []
    wrong_samples = []
    
    with torch.no_grad():
        for idx in indices:
            # Lấy 1 ảnh từ dataset
            inputs, labels_fruit, labels_qual = dataset[idx]
            
            # Thêm chiều batch_size=1
            inputs_batch = inputs.unsqueeze(0).to(model.device)
            
            # Quét qua model
            outputs_fruit, outputs_qual = model(inputs_batch)
            p_fruit = outputs_fruit.argmax(1).item()
            p_qual = outputs_qual.argmax(1).item()
            
            # Kiểm tra xem có dự đoán đúng hoàn toàn hay không
            is_correct = (p_fruit == labels_fruit.item() and p_qual == labels_qual.item())
            
            sample_data = (inputs, labels_fruit.item(), labels_qual.item(), p_fruit, p_qual)
            
            # Lưu lại vào danh sách tương ứng
            if is_correct and len(correct_samples) < num_correct:
                correct_samples.append(sample_data)
            elif not is_correct and len(wrong_samples) < num_wrong:
                wrong_samples.append(sample_data)
                
            # Đủ chỉ tiêu số lượng (đúng và sai) thì dừng
            if len(correct_samples) == num_correct and len(wrong_samples) == num_wrong:
                break

    # Gộp 2 danh sách lại để vẽ
    samples_to_show = correct_samples + wrong_samples
    
    if not samples_to_show:
        print("Không có ảnh nào để hiển thị.")
        return
        
    num_images = len(samples_to_show)
    num_rows = (num_images + 2) // 3
    fig = plt.figure(figsize=(15, 6 * num_rows))
    
    for i, (img_tensor, t_fruit, t_qual, p_fruit, p_qual) in enumerate(samples_to_show):
        ax = plt.subplot(num_rows, 3, i + 1)
        ax.axis('off')
        
        # Denormalize ảnh
        img = img_tensor.cpu().numpy().transpose((1, 2, 0))
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)
        
        true_f_name = id2fruit[t_fruit]
        true_q_name = id2fresh[t_qual]
        pred_f_name = id2fruit[p_fruit]
        pred_q_name = id2fresh[p_qual]
        
        title = f'Thực tế: {true_q_name} {true_f_name}\nDự đoán: {pred_q_name} {pred_f_name}'
        
        # Đúng = Xanh, Sai = Đỏ
        color = 'green' if (t_fruit == p_fruit and t_qual == p_qual) else 'red'
        ax.set_title(title, color=color, fontweight='bold')
        
        plt.imshow(img)
        
    plt.tight_layout()
    plt.show()

# Chạy hàm: Yêu cầu lấy 3 ảnh dự đoán đúng, 3 ảnh dự đoán sai 
# Truyền 'test' (dataset) vào thay vì test_loader để dễ dàng random index
visualize_random_and_wrong_predictions(model, test, num_correct=3, num_wrong=3)


## Other method

### Histogram RGB

In [ ]:
def readImgRGB(img_path, bins=256):
  img = cv2.imread(img_path)
  if img is None:
    return None
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

  feat_vector = []
  for i in range(3):
    hist = cv2.calcHist([img], [i], None, [bins], [0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    feat_vector.extend(hist)

  feat_vector = np.array(feat_vector)
  feat_vector = cv2.normalize(feat_vector, feat_vector).flatten()
  return feat_vector

### Histogram HSV

In [ ]:
def readImgHSV(img_path, bins=(8, 8, 8)):
  img = cv2.imread(img_path)
  if img is None:
    return None
  img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

  feat_vector = []
  hist = cv2.calcHist([img], [0, 1, 2], None, bins, [0, 180, 0, 256, 0, 256])
  hist = cv2.normalize(hist, hist).flatten()
  feat_vector.extend(hist)

  feat_vector = np.array(feat_vector)
  feat_vector = cv2.normalize(feat_vector, feat_vector).flatten()
  return feat_vector

### RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from xgboost import XGBClassifier

In [ ]:
def preprocessing_features(X_train, X_val, X_test):
  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)
  X_val_scaled = scaler.transform(X_val)
  X_test_scaled = scaler.transform(X_test)
  return X_train_scaled, X_val_scaled, X_test_scaled

In [ ]:
def classify(img_paths, labels_list, method='RF', img='RGB', task='fruit'):
    """
    Train model trên dữ liệu đã chia sẵn.
    img_paths: list đường dẫn ảnh (train)
    labels_list: list labels tương ứng
    """
    X_train, y_train = [], []
    
    for path, label in zip(img_paths, labels_list):
        if img == 'RGB':
            feat = readImgRGB(path)
        elif img == 'HSV':
            feat = readImgHSV(path)
            
        if feat is not None:
            X_train.append(feat)
            
            if task == 'fruit':
                y_train.append(label[0])  # label[0] là fruit_id
            elif task == 'quality':
                y_train.append(label[1])  # label[1] là fresh_id
    
    if method == 'RF':
        clf = RandomForestClassifier()
        params = {
            'n_estimators': [50, 100],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5],
            'min_samples_leaf': [1, 2],
            'max_features': ['sqrt', 'log2']
        }
        grid_search = GridSearchCV(estimator=clf, param_grid=params, cv=5, n_jobs=-1, verbose=1)
        grid_search.fit(X_train, y_train)
        best_rf = grid_search.best_estimator_
        print("Best RF Hyperparameters:", grid_search.best_params_)
        return best_rf
        
    elif method == 'XGB':
        clf = XGBClassifier(device='cuda', use_label_encoder=False, eval_metric='mlogloss')
        params = {
            'n_estimators': [50, 100, 200],
            'max_depth': [3, 6],
            'learning_rate': [0.05, 0.1],
            'subsample': [0.8, 1.0]
        }
        grid_search = GridSearchCV(estimator=clf, param_grid=params, cv=5, n_jobs=-1, verbose=1)
        grid_search.fit(X_train, y_train)
        best_xgb = grid_search.best_estimator_
        print("Best XGB Hyperparameters:", grid_search.best_params_)
        return best_xgb


In [ ]:
# ===== TRAIN ML MODELS CHỈ TRÊN TRAIN DATA =====
best_cls_fruit = classify(train_paths, train_labels, method='XGB', img='RGB', task='fruit')
print("Best Classifier (Fruit):", best_cls_fruit)

print("\n" + "="*50 + "\n")

best_cls_quality = classify(train_paths, train_labels, method='XGB', img='RGB', task='quality')
print("Best Classifier (Quality):", best_cls_quality)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

def evaluate_ml_models(img_paths, labels_list, model_fruit, model_quality, img='RGB'):
    """
    Evaluate ML models trên tập dữ liệu riêng (val hoặc test).
    """
    X_eval, y_true_fruit, y_true_quality = [], [], []
    for path, label in zip(img_paths, labels_list):
        if img == 'RGB':
            feat = readImgRGB(path)
        elif img == 'HSV':
            feat = readImgHSV(path)
            
        if feat is not None:
            X_eval.append(feat)
            y_true_fruit.append(label[0])
            y_true_quality.append(label[1])
            
    y_pred_fruit = model_fruit.predict(X_eval)
    y_pred_quality = model_quality.predict(X_eval)
    
    acc_fruit = accuracy_score(y_true_fruit, y_pred_fruit)
    acc_quality = accuracy_score(y_true_quality, y_pred_quality)

    print("    ĐÁNH GIÁ MÔ HÌNH ML (RGB Features)")
    print(f"    [+] Fruit Accuracy  : {acc_fruit:.4f}")
    print(f"    [+] Quality Accuracy: {acc_quality:.4f}\n")
    
    print("    --- 1. Báo cáo Chi tiết loại Trái Cây ---")
    print(classification_report(y_true_fruit, y_pred_fruit, target_names=list(fruit2id.keys())))
    
    print("    --- 2. Báo cáo Chi tiết Độ Tươi Khô ---")
    print(classification_report(y_true_quality, y_pred_quality, target_names=['Fresh (0)', 'Rotten/Spoiled (1)']))

# ===== EVALUATE TRÊN TEST DATA (DATA MODEL CHƯA TỪNG THẤY) =====
print("=" * 60)
print("  EVALUATE TRÊN TEST SET (dữ liệu model chưa từng thấy)")
print("=" * 60)
evaluate_ml_models(test_paths, test_labels, best_cls_fruit, best_cls_quality, img='RGB')


In [ ]:
# ===== TRAIN ML MODELS CHỈ TRÊN TRAIN DATA =====
best_cls_fruit_RF_RGB = classify(train_paths, train_labels, method='RF', img='RGB', task='fruit')
print("Best Classifier (Fruit):", best_cls_fruit_RF_RGB)

print("\n" + "="*50 + "\n")

best_cls_quality_RF_RGB = classify(train_paths, train_labels, method='RF', img='RGB', task='quality')
print("Best Classifier (Quality):", best_cls_quality_RF_RGB)


In [ ]:
evaluate_ml_models(test_paths, test_labels, best_cls_fruit, best_cls_quality, img='RGB')


In [ ]:
# ===== TRAIN ML MODELS CHỈ TRÊN TRAIN DATA =====
best_cls_fruit = classify(train_paths, train_labels, method='XGB', img='HSV', task='fruit')
print("Best Classifier (Fruit):", best_cls_fruit)

print("\n" + "="*50 + "\n")

best_cls_quality = classify(train_paths, train_labels, method='XGB', img='HSV', task='quality')
print("Best Classifier (Quality):", best_cls_quality)


In [ ]:

joblib.dump(best_xgb_fruit, 'xgb_fruit_model.pkl')
joblib.dump(best_xgb_quality, 'xgb_quality_model.pkl')
print("[+] Đã xuất file: xgb_fruit_model.pkl và xgb_quality_model.pkl\n")

In [ ]:
evaluate_ml_models(test_paths, test_labels, best_cls_fruit, best_cls_quality, img='HSV')


In [ ]:
# ===== TRAIN ML MODELS CHỈ TRÊN TRAIN DATA =====
best_cls_fruit = classify(train_paths, train_labels, method='RF', img='HSV', task='fruit')
print("Best Classifier (Fruit):", best_cls_fruit)

print("\n" + "="*50 + "\n")

best_cls_quality = classify(train_paths, train_labels, method='RF', img='HSV', task='quality')
print("Best Classifier (Quality):", best_cls_quality)


In [ ]:
evaluate_ml_models(test_paths, test_labels, best_cls_fruit, best_cls_quality, img='HSV')

In [ ]:
# Dự đoán cho ảnh đơn
single_img_path = os.path.join(NOTEBOOK_DIR, "meo-cam-1.ipg")  # theo yêu cầu gốc
if not os.path.isfile(single_img_path):
    single_img_path = os.path.join(NOTEBOOK_DIR, "meo-cam-1.jpg")  # fallback đúng đuôi file

assert os.path.isfile(single_img_path), f"Không tìm thấy ảnh: {single_img_path}"
assert 'model' in globals(), "Chưa có model trong kernel. Hãy train/chạy cell model trước."

def predict_single_image_resnet(model, img_path, transform):
    model.eval()
    bgr = cv2.imread(img_path)
    if bgr is None:
        raise FileNotFoundError(f"Không thể đọc ảnh: {img_path}")

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    tensor = transform(image=rgb)['image'].unsqueeze(0).to(model.device)

    with torch.no_grad():
        out_fruit, out_quality = model(tensor)
        prob_fruit = torch.softmax(out_fruit, dim=1).squeeze(0)
        prob_quality = torch.softmax(out_quality, dim=1).squeeze(0)

        pred_fruit_id = int(torch.argmax(prob_fruit).item())
        pred_quality_id = int(torch.argmax(prob_quality).item())

    pred_fruit_name = id2fruit[pred_fruit_id]
    pred_quality_name = id2fresh[pred_quality_id]

    print(f"Image: {img_path}")
    print(f"Predicted Fruit  : {pred_fruit_name} (conf={prob_fruit[pred_fruit_id].item():.4f})")
    print(f"Predicted Quality: {pred_quality_name} (conf={prob_quality[pred_quality_id].item():.4f})")

    plt.figure(figsize=(5, 5))
    plt.imshow(rgb)
    plt.title(f"{pred_quality_name} {pred_fruit_name}")
    plt.axis('off')
    plt.show()

predict_single_image_resnet(model, single_img_path, val_trans)

In [ ]:
# Predict ảnh đơn bằng RF + RGB features
single_img_path_rf = os.path.join(NOTEBOOK_DIR, "meo-cam-1.jpg")

assert os.path.isfile(single_img_path_rf), f"Không tìm thấy ảnh: {single_img_path_rf}"
assert 'best_cls_fruit' in globals() and 'best_cls_quality' in globals(), (
    "Chưa có model RF trong kernel. Hãy chạy cell train RF trước."
)

feat = readImgRGB(single_img_path_rf)
if feat is None:
    raise FileNotFoundError(f"Không thể đọc ảnh: {single_img_path_rf}")

pred_fruit_id = int(best_cls_fruit_RF_RGB.predict([feat])[0])
pred_quality_id = int(best_cls_quality_RF_RGB.predict([feat])[0])

# Lấy confidence nếu model hỗ trợ predict_proba
fruit_conf = None
quality_conf = None
if hasattr(best_cls_fruit_RF_RGB, 'predict_proba'):
    fruit_conf = float(np.max(best_cls_fruit_RF_RGB.predict_proba([feat])[0]))
if hasattr(best_cls_quality_RF_RGB, 'predict_proba'):
    quality_conf = float(np.max(best_cls_quality_RF_RGB.predict_proba([feat])[0]))

id2fruit = {v: k for k, v in fruit2id.items()}
id2quality = {0: 'fresh', 1: 'rotten'}

pred_fruit_name = id2fruit.get(pred_fruit_id, str(pred_fruit_id))
pred_quality_name = id2quality.get(pred_quality_id, str(pred_quality_id))

print(f"Image: {single_img_path_rf}")
if fruit_conf is not None:
    print(f"Predicted Fruit  : {pred_fruit_name} (conf={fruit_conf:.4f})")
else:
    print(f"Predicted Fruit  : {pred_fruit_name}")

if quality_conf is not None:
    print(f"Predicted Quality: {pred_quality_name} (conf={quality_conf:.4f})")
else:
    print(f"Predicted Quality: {pred_quality_name}")

img_rgb = cv2.cvtColor(cv2.imread(single_img_path_rf), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(5, 5))
plt.imshow(img_rgb)
plt.title(f"RF-RGB: {pred_quality_name} {pred_fruit_name}")
plt.axis('off')
plt.show()